# Explorer 06 - Networks and communities

**Purpose.** Build the collaboration, credit, and artist-label graphs; compute centrality; detect communities with Leiden; compare them with genre tags.

**Feeds main report section:** 5.5 RQ4 - Communities

**Modules used:** `vinyl.networks`

**Tables read:** `release_artists`, `release_credits`, `release_labels`, `release_genres`

This is a working notebook. It can be messy; anything that goes into the report is re-run from `main.ipynb` through the same functions in `vinyl/`.

In [ ]:
# Make the `vinyl` package importable from the explorers/ folder.
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "explorers":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

%load_ext autoreload
%autoreload 2

import time

import pandas as pd
from pyspark.sql import functions as F
from vinyl import config, ingest

spark = ingest.get_spark(app_name="vinyl-06-networks")
from vinyl import networks
from vinyl import labels as label_power

In [ ]:
snapshot = config.MAIN_SNAPSHOT  # "2026"
release_artists = ingest.read_table(spark, snapshot, "release_artists")
release_credits = ingest.read_table(spark, snapshot, "release_credits")
release_labels = ingest.read_table(spark, snapshot, "release_labels")
release_genres = ingest.read_table(spark, snapshot, "release_genres")
release_styles = ingest.read_table(spark, snapshot, "release_styles")
artists = ingest.read_table(spark, snapshot, "artists")      # names for display only
labels = ingest.read_table(spark, snapshot, "labels")

## 1. Edge lists (Spark)

Minimum edge weights come from `config.GRAPH_MIN_WEIGHT`, chosen so each graph fits on the driver (node and edge counts are in the `config` comment). Self-released labels (`Not On Label` and its variants) are dropped from the artist-label graph, otherwise they become one giant hub.

In [ ]:
self_released = label_power.self_released_label_ids(labels).cache()
print("self-released label ids dropped from the artist-label graph:", f"{self_released.count():,}")

w = config.GRAPH_MIN_WEIGHT
min_weight = {"collaboration": w["collaboration"], "credit": w["credit"], "artist-label": w["artist_label"]}
collab = networks.collaboration_edges(release_artists, min_weight=min_weight["collaboration"], max_artists_per_release=20).cache()
credit = networks.credit_edges(release_artists, release_credits, roles=config.PRODUCTION_ROLES, min_weight=min_weight["credit"]).cache()
artist_label = networks.artist_label_edges(release_artists, release_labels, min_weight=min_weight["artist-label"], exclude_label_ids=self_released).cache()
for name, edges in [("collaboration", collab), ("credit", credit), ("artist-label", artist_label)]:
    print(f"{name:14s} min_weight {min_weight[name]}  edges {edges.count():,}")

In [ ]:
collab.groupBy("weight").count().orderBy("weight").show(20)   # weight distribution -> choose min_weight

## 2. Graphs and centrality (driver)

The artist-label graph uses prefixed ids (`a123` / `l123`) because artist and label ids overlap as numbers.

In [ ]:
edge_tables = {
    "collaboration": collab.toPandas(),
    "credit": credit.toPandas(),
    "artist-label": networks.prefixed_bipartite(artist_label).toPandas(),
}
graphs = {k: networks.to_igraph(v) for k, v in edge_tables.items()}
for k, g in graphs.items():
    print(f"{k:14s} nodes {g.vcount():>10,}  edges {g.ecount():>10,}")

# Names only for the nodes in the graphs, not all 10 M artists.
names = {
    "collaboration": networks.node_names(artists, ids=graphs["collaboration"].vs["name"]),
    "credit": networks.node_names(artists, ids=graphs["credit"].vs["name"]),
    "artist-label": networks.node_names(artists, labels, prefixed=True, ids=graphs["artist-label"].vs["name"]),
}
centrality = {k: networks.centrality_table(g, betweenness=False) for k, g in graphs.items()}
for k in graphs:
    print(f"\n{k}: top 20 by PageRank")
    display(centrality[k].nlargest(20, "pagerank").merge(names[k], on="node", how="left"))

## 3. Leiden communities

In [ ]:
communities = {k: networks.leiden_communities(g, resolution=1.0, seed=42) for k, g in graphs.items()}
for k, c in communities.items():
    sizes = c["community"].value_counts()
    print(f"{k:14s} communities {len(sizes):>8,}  >=50 nodes {(sizes >= 50).sum():>5,}  largest {sizes.iloc[0]:>8,}  modularity {c.attrs['modularity']:.4f}")

## 4. Do communities match genre and style tags?

Each artist gets the genre (and the style) on most of their releases. The credit graph also holds credited people who are never a main artist; they have no tag, so agreement there covers only the nodes that have one. In the artist-label graph agreement covers the artist nodes only.

In [ ]:
# Dominant tags only for artists that are in a graph, to keep the driver table small.
artist_ids = sorted(
    set(networks.graph_artist_ids(graphs["collaboration"]))
    | set(networks.graph_artist_ids(graphs["credit"]))
    | set(networks.graph_artist_ids(graphs["artist-label"], prefixed=True))
)
in_graphs = spark.createDataFrame([(i,) for i in artist_ids], "artist_id long")
graph_artists = release_artists.join(in_graphs, "artist_id")
dominant = {
    "genre": networks.artist_dominant_tag(graph_artists, release_genres, "genre").toPandas().rename(columns={"artist_id": "node"}),
    "style": networks.artist_dominant_tag(graph_artists, release_styles, "style").toPandas().rename(columns={"artist_id": "node"}),
}
# Artist-label nodes are prefixed, so prefix the tag table too (label nodes then have no tag).
node_tags = {
    (k, tag): (t.assign(node="a" + t["node"].astype(str)) if k == "artist-label" else t)
    for k in graphs for tag, t in dominant.items()
}

rows = []
for k, g in graphs.items():
    n_artist_nodes = len(networks.graph_artist_ids(g, prefixed=(k == "artist-label")))
    for tag in ("genre", "style"):
        agreement_row = networks.community_tag_agreement(communities[k], node_tags[(k, tag)], tag_col=tag)
        rows.append({"graph": k, "tag": tag, "n_artist_nodes": n_artist_nodes, **agreement_row})
agreement = pd.DataFrame(rows)
agreement["share_tagged"] = agreement["n_nodes"] / agreement["n_artist_nodes"]
display(agreement)

## 5. Largest communities

Top 10 communities per graph with at least 50 nodes: the five highest-PageRank nodes, the three most common tags and the tag purity (share of the community's tagged nodes that carry the most common tag).

In [ ]:
for k in graphs:
    for tag in ("genre", "style"):
        desc = networks.describe_communities(
            communities[k], centrality[k], names[k], node_tags=node_tags[(k, tag)],
            top_n=5, min_size=50, tag_col=tag,
        ).head(10)
        print(f"\n{k}: top 10 communities, dominant {tag}")
        display(desc)

## 6. Resolution sweep

How stable are the communities when Leiden's resolution changes? `nmi_vs_reference` compares each run with resolution 1.0. The credit graph is only swept if the time estimated from the collaboration run (scaled by edge count) is under 5 minutes.

In [ ]:
sweeps = {}
start = time.perf_counter()
sweeps["collaboration"] = networks.resolution_sweep(graphs["collaboration"], resolutions=(0.5, 1.0, 2.0), seed=42, reference=1.0)
collab_seconds = time.perf_counter() - start
print(f"collaboration sweep: {collab_seconds:.1f} s")
display(sweeps["collaboration"])

estimate = collab_seconds * graphs["credit"].ecount() / graphs["collaboration"].ecount()
print(f"estimated credit sweep: {estimate:.0f} s (limit 300 s)")
if estimate < 300:
    start = time.perf_counter()
    sweeps["credit"] = networks.resolution_sweep(graphs["credit"], resolutions=(0.5, 1.0, 2.0), seed=42, reference=1.0)
    print(f"credit sweep: {time.perf_counter() - start:.1f} s")
    display(sweeps["credit"])
else:
    print("credit sweep skipped")

## 7. Summary table

One row per graph and tag type. The report quotes this table.

In [ ]:
graph_stats = pd.DataFrame({
    "graph": list(graphs),
    "min_weight": [min_weight[k] for k in graphs],
    "nodes": [g.vcount() for g in graphs.values()],
    "edges": [g.ecount() for g in graphs.values()],
    "communities": [communities[k]["community"].nunique() for k in graphs],
    "modularity": [communities[k].attrs["modularity"] for k in graphs],
})
summary = graph_stats.merge(agreement, on="graph")
display(summary[["graph", "min_weight", "nodes", "edges", "communities", "modularity", "tag", "n_nodes", "share_tagged", "nmi", "ari", "purity"]].round(4))

## Decisions to carry into `main.ipynb`

Fill this in when the exploration is done. Each line should end up as a sentence in the report's methodology or limitations.

- [ ] `min_weight` and `max_artists_per_release` for each graph, and resulting node/edge counts.
- [ ] Leiden resolution, seed, and modularity.
- [ ] Agreement scores (NMI, ARI, purity) against genre and against style.
- [ ] Names for the communities shown in the report.